In [22]:
import numpy as np
import matplotlib.pyplot as plt
from debugpy.launcher import output
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [24]:
XX = 1

with open(f'data/dane{XX}.txt', 'r') as file:
    data = file.readlines()

x_data, y_data = zip(*[
    map(float, line.split())
    for line in data
])

scaler = MinMaxScaler()
x_data_normalized = scaler.fit_transform(np.array(x_data).reshape(-1, 1))
y_data_normalized = scaler.fit_transform(np.array(y_data).reshape(-1, 1))

X_train, X_test, Y_train, Y_test = train_test_split(x_data_normalized, y_data_normalized, test_size=0.2,
                                                    random_state=42)

In [ ]:
class NeuralNetworkCustom:
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.weights_input_hidden = np.random.randn(self.input_size, self.hidden_size)
        self.weights_hidden_output = np.random.randn(self.hidden_size, self.output_size)

        self.bias_hidden = np.zeros(self.hidden_size)
        self.bias_output = np.zeros(self.output_size)

        self.history_mse = []
        self.history_r2 = []

    def tanh(self, x):
        return np.tanh(x)

    def tanh_derivative(self, x):
        return 1.0 - np.tanh(x) ** 2

    def forward(self, X):
        hidden_input = np.dot(X, self.weights_input_hidden) + self.bias_hidden
        hidden_input = self.tanh(hidden_input)

        output_input = np.dot(hidden_input, self.weights_hidden_output) + self.bias_output
        return self.tanh(output_input)

    def backprop(self, X, y, output, learning_rate, reg_lambda):
        output_error = y - output

        hidden_output = self.tanh(np.dot(X, self.weights_input_hidden) + self.bias_hidden)
        gradient_hidden_output = np.dot(hidden_output.T, output_error)

        hidden_error = np.dot(output_error, self.weights_hidden_output.T)
        hidden_error *= self.tanh_derivative(hidden_output)

        gradient_input_hidden = np.dot(X.T, hidden_error)

        self.weights_hidden_output += (gradient_hidden_output - reg_lambda * self.weights_hidden_output) * learning_rate
        self.weights_input_hidden += (gradient_input_hidden - reg_lambda * self.weights_input_hidden) * learning_rate

        self.bias_output += np.sum(output_error, axis=0) * learning_rate
        self.bias_hidden += np.sum(hidden_error, axis=0) * learning_rate

    def train(self, X, y, epochs, learning_rate, reg_lambda):
        print("Neural network training starts...")

        for epoch in range(1, epochs):
            output = self.forward(X)
            self.backprop(X, y, output, learning_rate, reg_lambda)

            mse = mean_squared_error(y, output)
            r2 = r2_score(y, output)

            self.history_mse.append(mse)
            self.history_r2.append(r2)

            if epoch % (epochs // 10) == 0:
                print(f'[%] {epoch / epochs}')

            if r2 >= 0.95:
                print("Training stopped, R2 score reached 0.95")
                break






